# ResNet50 transfer learning (ImageNet weights)

Based on **`model_template.ipynb`**: same Open Images classes (**Egg (Food)**, **Chicken**, **Balloon**), FiftyOne export, and the same **80/20** split and training loop as **`train_resnet50_scratch.py`**, with **ImageNet-pretrained** ResNet50. **`NUM_EPOCHS` is 10** to match Project C’s comparison over the first 10 epochs.

**Local:** set the working directory to the **project root** (folder containing `src/`) or to **`src/`**.

**Google Colab:** follow the **Google Colab** section below (GPU runtime, clone or upload repo, then run all cells).

### Google Colab

1. **Runtime → Change runtime type → GPU** (recommended for ResNet50 training).
2. **Get the full repo** on the VM so `src/download_images.py` exists. Either:
   - **Clone** your repository into `/content` (edit the URL in the next cell), then run `%cd` into the project root, **or**
   - Upload the project folder to `/content` and `%cd` into it.
3. **Open this notebook from that project root** (or keep `%cd` there before running the cells below). Dataset cache will live under `/content/data`; metrics/plots under `<project>/results/`.

PyTorch with CUDA is **usually preinstalled** on Colab; the install cell only adds FiftyOne and other dependencies.


In [ ]:
# --- Colab: clone repo (set URL), then cd to project root ---
import os
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Set your repository URL (use a public repo or a token URL for private).
    REPO_URL = "https://github.com/YOUR_USER/image_classification.git"
    CLONE_DIR = Path("/content/image_classification")

    if not (CLONE_DIR / "src" / "download_images.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            check=True,
        )
    else:
        print("Repo already present at", CLONE_DIR)

    os.chdir(CLONE_DIR)
    print("cwd:", Path.cwd())
else:
    print("Not on Colab — skip clone; run from project root or src/.")


### Installs

In [ ]:
%pip install -q fiftyone tqdm pandas matplotlib seaborn numpy scikit-learn
%pip install -q -U eta

import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Repository paths

In [ ]:
import os
import sys
from pathlib import Path


def _repo_and_src():
    cwd = Path.cwd().resolve()
    roots = [cwd]
    colab_guess = Path("/content/image_classification")
    if colab_guess not in roots:
        roots.append(colab_guess)

    for root in roots:
        if (root / "src" / "download_images.py").is_file():
            return root, root / "src"
        if (root / "download_images.py").is_file():
            return root.parent, root

    raise FileNotFoundError(
        "Could not find download_images.py. On Colab, run the clone cell and cd to the project root."
    )


REPO_ROOT, SRC_DIR = _repo_and_src()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_DIR:", SRC_DIR)

### Download dataset and 80/20 split

In [ ]:
import random
import shutil

from download_images import create_classification_dataset_from_openimages

TARGET_CLASSES = ["Egg (Food)", "Chicken", "Balloon"]
MAX_IMAGES_PER_CLASS = 500
MAX_SOURCE_SAMPLES = 2000


def detect_env() -> str:
    if Path("/kaggle").exists():
        return "kaggle"
    if Path("/content").exists():
        return "colab"
    return "local"


def resolve_paths():
    env = detect_env()
    if env == "kaggle":
        base_data_dir = Path("/kaggle/working/data")
    elif env == "colab":
        base_data_dir = Path("/content/data")
    else:
        base_data_dir = REPO_ROOT / "data"
    data_root = base_data_dir / "openimages_subset" / "classification"
    split_root = base_data_dir / "openimages_subset_split"
    results_root = REPO_ROOT / "results"
    base_data_dir.mkdir(parents=True, exist_ok=True)
    results_root.mkdir(parents=True, exist_ok=True)
    return env, data_root, split_root, results_root


def get_class_counts(data_root: Path, class_names):
    counts = {}
    for class_name in class_names:
        class_dir = data_root / class_name
        if class_dir.exists():
            counts[class_name] = len([p for p in class_dir.iterdir() if p.is_file()])
        else:
            counts[class_name] = 0
    return counts


def ensure_dataset(data_root: Path, class_names, max_images_per_class: int, max_samples: int):
    if data_root.exists():
        class_counts = get_class_counts(data_root, class_names)
        if all(count == max_images_per_class for count in class_counts.values()):
            print("Using existing dataset at", data_root)
            print("Existing class counts:", class_counts)
            return
        print("Existing dataset does not exactly match requested sample size. Rebuilding...")
        print("Existing class counts:", class_counts)
        shutil.rmtree(data_root)

    create_classification_dataset_from_openimages(
        class_names=class_names,
        output_dir=str(data_root),
        max_images_per_class=max_images_per_class,
        max_samples=max_samples,
    )


def create_train_test_split(source_dir: Path, split_root: Path, train_ratio: float = 0.8, seed: int = 42):
    rng = random.Random(seed)
    if split_root.exists():
        shutil.rmtree(split_root)
    train_dir = split_root / "train"
    test_dir = split_root / "test"
    summary = {}
    for class_dir in sorted([p for p in source_dir.iterdir() if p.is_dir()]):
        files = sorted([p for p in class_dir.iterdir() if p.is_file()])
        if not files:
            continue
        rng.shuffle(files)
        split_idx = int(len(files) * train_ratio)
        train_files = files[:split_idx]
        test_files = files[split_idx:]
        class_train_dir = train_dir / class_dir.name
        class_test_dir = test_dir / class_dir.name
        class_train_dir.mkdir(parents=True, exist_ok=True)
        class_test_dir.mkdir(parents=True, exist_ok=True)
        for src in train_files:
            shutil.copy2(src, class_train_dir / src.name)
        for src in test_files:
            shutil.copy2(src, class_test_dir / src.name)
        summary[class_dir.name] = {"train": len(train_files), "test": len(test_files)}
    return train_dir, test_dir, summary


env, data_root, split_root, results_root = resolve_paths()
print("Environment:", env)
print("Data root:", data_root)

ensure_dataset(
    data_root,
    class_names=TARGET_CLASSES,
    max_images_per_class=MAX_IMAGES_PER_CLASS,
    max_samples=MAX_SOURCE_SAMPLES,
)

train_dir, test_dir, split_summary = create_train_test_split(
    data_root, split_root, train_ratio=0.8, seed=42
)
print("Split summary:", split_summary)

### Train ResNet50 (ImageNet transfer)

Writes `results/resnet50_transfer_summary.csv`, `resnet50_transfer_history.csv`, `resnet50_transfer_curves.png`.

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_WORKERS = 0 if os.name == "nt" else 4

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| device:", device)

train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
test_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset), "| Test:", len(test_dataset))

try:
    from torchvision.models import ResNet50_Weights

    weights = ResNet50_Weights.IMAGENET1K_V1
except Exception:
    weights = "IMAGENET1K_V1"

try:
    model = models.resnet50(weights=weights)
except TypeError:
    model = models.resnet50(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_parameters = int(sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", num_parameters)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_examples += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return total_loss / total_examples, accuracy_score(all_labels, all_preds)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * images.size(0)
            total_examples += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return total_loss / total_examples, accuracy_score(all_labels, all_preds)


history = {
    "epoch_time_seconds": [],
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
}

t0 = time.time()
for epoch in range(NUM_EPOCHS):
    t_ep = time.time()
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    dt = time.time() - t_ep
    history["epoch_time_seconds"].append(dt)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"test_loss={test_loss:.4f} acc={test_acc:.4f} | {dt:.1f}s"
    )
total_training_time = time.time() - t0

final_test_loss, final_test_acc = evaluate(model, test_loader, criterion, device)
final_train_acc = history["train_acc"][-1]
print(f"Final test accuracy: {final_test_acc * 100:.2f}%")
print(f"Total training time: {total_training_time:.1f}s")

summary_path = results_root / "resnet50_transfer_summary.csv"
history_path = results_root / "resnet50_transfer_history.csv"
figure_path = results_root / "resnet50_transfer_curves.png"

pd.DataFrame(
    [
        {
            "model": "resnet50_transfer_imagenet",
            "seed": SEED,
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "classes": ", ".join(train_dataset.classes),
            "train_samples": len(train_dataset),
            "test_samples": len(test_dataset),
            "final_train_accuracy": final_train_acc,
            "final_test_accuracy": final_test_acc,
            "final_test_loss": final_test_loss,
            "num_parameters": num_parameters,
            "total_training_time_seconds": total_training_time,
            "average_epoch_time_seconds": sum(history["epoch_time_seconds"])
            / len(history["epoch_time_seconds"]),
        }
    ]
).to_csv(summary_path, index=False)

pd.DataFrame(
    {
        "epoch": list(range(1, NUM_EPOCHS + 1)),
        "epoch_time_seconds": history["epoch_time_seconds"],
        "train_loss": history["train_loss"],
        "train_acc": history["train_acc"],
        "test_loss": history["test_loss"],
        "test_acc": history["test_acc"],
    }
).to_csv(history_path, index=False)

epochs = range(1, NUM_EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history["train_loss"], label="Train")
ax[0].plot(epochs, history["test_loss"], label="Test")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title(f"Transfer: loss ({NUM_EPOCHS} epochs)")
ax[0].legend()
ax[1].plot(epochs, history["train_acc"], label="Train")
ax[1].plot(epochs, history["test_acc"], label="Test")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_title(f"Transfer: accuracy ({NUM_EPOCHS} epochs)")
ax[1].legend()
plt.tight_layout()
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", summary_path)
print("Saved:", history_path)
print("Saved:", figure_path)

### Scratch vs transfer (first 10 epochs)

Requires `results/resnet50_scratch_history.csv` from **`python src/train_resnet50_scratch.py`** and `resnet50_transfer_history.csv` from the training cell above. Saves `results/scratch_vs_transfer_first10.png`.

In [ ]:
COMPARE_FIRST_N = 10
scratch_csv = results_root / "resnet50_scratch_history.csv"
transfer_csv = results_root / "resnet50_transfer_history.csv"
out_path = results_root / "scratch_vs_transfer_first10.png"

if not scratch_csv.is_file() or not transfer_csv.is_file():
    missing = [p.name for p in (scratch_csv, transfer_csv) if not p.is_file()]
    raise FileNotFoundError(
        f"Missing history CSV(s): {missing}. Run train_resnet50_scratch.py and the transfer training cell first."
    )

df_s = pd.read_csv(scratch_csv).head(COMPARE_FIRST_N)
df_t = pd.read_csv(transfer_csv).head(COMPARE_FIRST_N)
n = min(len(df_s), len(df_t), COMPARE_FIRST_N)
df_s = df_s.iloc[:n]
df_t = df_t.iloc[:n]
ep = df_s["epoch"].values

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, df_s["test_loss"], label="Scratch (test)", marker="o", markersize=3)
ax[0].plot(ep, df_t["test_loss"], label="Transfer (test)", marker="o", markersize=3)
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title(f"Test loss: scratch vs transfer (first {n} epochs)")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(ep, df_s["test_acc"], label="Scratch (test)", marker="o", markersize=3)
ax[1].plot(ep, df_t["test_acc"], label="Transfer (test)", marker="o", markersize=3)
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_title(f"Test accuracy: scratch vs transfer (first {n} epochs)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out_path)